# Module 3 · Explainability and drift monitoring
    **Outcomes:** (e) explain a prediction with SHAP, in plain language; (f) detect drift with PSI and KS, and set thresholds.

> **How to use this notebook.** Run cells top to bottom with **Shift + Enter**.
> Cells marked **✏️ TODO** are yours to complete. **🏁 Checkpoint** cells tell you when a module is done.
> Stuck? Ask your neighbour first, then the trainer. Solutions are released after the session.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.stats import ks_2samp
from sklearn.ensemble import IsolationForest

from course_utils import check_drift_table, check_narrative, check_psi, make_credit_data, make_drift_data

model, X_train, X_test, y_train, y_test = make_credit_data()
print(f"Gradient boosting credit model · test accuracy {model.score(X_test, y_test):.2f}")

## A · SHAP: fair credit for every feature (20 min)
`f(x) = E[f(X)] + Σ φᵢ`: each feature's contribution adds up to the prediction (in log-odds).
**Global** view first: what drives the whole portfolio? Red is a high value, right means more risk.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
base_value = float(np.ravel(explainer.expected_value)[0])
shap.summary_plot(shap_values, X_test)

**Local** view: why did *this* applicant get *this* score?

In [ ]:
i = 0   # applicant 0, the one on the slides
shap.waterfall_plot(shap.Explanation(values=shap_values[i], base_values=base_value,
                                     data=X_test.iloc[i], feature_names=list(X_test.columns)))

contrib = pd.DataFrame({"value": X_test.iloc[i].round(2), "shap": shap_values[i].round(3)})
contrib = contrib.reindex(contrib["shap"].abs().sort_values(ascending=False).index)
probability = 1 / (1 + np.exp(-(base_value + shap_values[i].sum())))
print(f"Applicant {i}: estimated default probability {probability:.0%}")
contrib

## B · Pair exercise: from numbers to a narrative (10 min)
Never hand a raw SHAP value to a stakeholder. Formula: **observed value → direction and size → one action that would change the decision.**

*Example for a different applicant:* "High card utilisation (87%) was the main reason for the decline: it pushed the estimated
default risk well above the approval line. Reducing utilisation below 30% would move the application to approval."

In [ ]:
# ✏️ TODO: three sentences about applicant 0 that a branch manager could act on (use the table above)
sentence_1 = ""   # the main driver and its value
sentence_2 = ""   # its direction and size, in plain words (no "SHAP", no "log-odds")
sentence_3 = ""   # one action or change that would move the decision

check_narrative([sentence_1, sentence_2, sentence_3])

Swap with your neighbour: **could a branch manager with no ML background act on it?**

## C · Detecting drift: PSI and the KS test (15 min)
One month of production data against the training month. Same simulation as the slides.

In [ ]:
train, prod = make_drift_data()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(train["credit_utilization"], bins=40, alpha=0.5, density=True, label="training")
ax.hist(prod["credit_utilization"], bins=40, alpha=0.5, density=True, label="production")
ax.set(title="credit_utilization: training vs production", xlabel="utilisation")
ax.legend()
plt.show()

**Population Stability Index:** cut the *training* data into 10 equal bins, then compare the share of rows per bin.
`PSI = Σ (Aᵢ − Eᵢ) · ln(Aᵢ / Eᵢ)` where E is the training share and A the production share.
Bands: **< 0.10 stable · 0.10–0.20 watch · ≥ 0.20 alert.**

In [ ]:
def psi(expected, actual, buckets=10):
    """Population Stability Index of `actual` (production) against `expected` (training)."""
    edges = np.percentile(expected, np.linspace(0, 100, buckets + 1))   # bins from the TRAINING quantiles
    edges[0], edges[-1] = -np.inf, np.inf                               # open ends: nothing falls off
    e = np.clip(np.histogram(expected, edges)[0] / len(expected), 1e-4, None)   # training share per bin
    a = np.clip(np.histogram(actual, edges)[0] / len(actual), 1e-4, None)       # production share per bin
    # ✏️ TODO: return PSI = Σ (a − e) · ln(a / e)      hint: np.sum, np.log
    return ...

check_psi(psi)

In [ ]:
rows = []
for col in train.columns:
    p = psi(train[col], prod[col])
    ks = ks_2samp(train[col], prod[col])
    # ✏️ TODO: "ALERT" if p is 0.20 or more, "WATCH" if 0.10 or more, otherwise "STABLE"
    status = ...
    rows.append({"feature": col, "psi": round(p, 3), "ks_pvalue": round(ks.pvalue, 4), "status": status})

drift = pd.DataFrame(rows).sort_values("psi", ascending=False)
print(drift.to_string(index=False), "\n")
check_drift_table(drift)

Look at `annual_income`: the KS p-value is tiny, yet PSI says stable. With thousands of rows KS flags
*any* difference; PSI tells you whether it is **big enough to act on**.

## D · Trainer demo: anomalies without labels (Isolation Forest)

In [ ]:
rng = np.random.default_rng(0)
cols = ["credit_utilization", "age"]
iso = IsolationForest(contamination=0.05, random_state=42).fit(train[cols].to_numpy())

normal = train[cols].sample(950, random_state=0).to_numpy()   # ordinary applicants
odd = np.column_stack([rng.uniform(0.85, 1.0, 50), rng.uniform(85, 110, 50)])   # 50 injected odd rows
live = np.vstack([normal, odd])
flagged = iso.predict(live) == -1
print(f"Injected 50 odd rows · flagged {flagged.sum()} in total · caught {flagged[-50:].sum()} of the 50")

plt.figure(figsize=(7, 4))
plt.scatter(live[~flagged, 0], live[~flagged, 1], s=8, alpha=0.4, label="normal")
plt.scatter(live[flagged, 0], live[flagged, 1], s=30, marker="x", color="crimson", label="flagged")
plt.xlabel("credit_utilization"); plt.ylabel("age"); plt.legend(); plt.title("Isolation Forest on live traffic")
plt.show()

`contamination=0.05` tells the forest to expect about 5% odd rows, so it also flags a few ordinary ones.
Tune it on real traffic so the alert rate is one a human will actually read.

### Monitoring cadence for the loan model
| Cadence | Check | Action |
|---|---|---|
| Every request | schema and ranges (Pydantic) | reject with HTTP 422 |
| Daily | volume, missing rate, anomaly rate | investigate if anomalies > 2× baseline |
| Weekly | PSI and KS per feature | watch at 0.1, alert at 0.2 |
| Monthly | AUC and recall on labelled loans | retrain if below the SLO |
| Quarterly | model review with risk | re-approve or retire |

### Quick quiz
1. What does SHAP additivity guarantee?
2. A feature has PSI = 0.27. What do you do?
3. Does Isolation Forest need labelled anomalies?